# Day 3 - Data Cleaning Topics

Welcome to the last day of the Intro to Python Bootcamp! You've learned a lot over the past couple days, starting with basic syntax and moving on to fully functional scripts that interact with external data. For this last day, we're going to take a look at some more practical topics and tasks that can be acheived with Python, including data cleaning, extraction, and visualiation. Then, you're going to be working on a short project where you will implement everyting you've learned.

## Examining Our Dataset
The data we will be working with for this session is a fictional dataset of registration survey responses for an upcoming health education workshop. Below are the details of the dataset:

### Data Dictionary: Workshop Participants Dataset
**Dataset Name:** `workshop_participants_east_coast.csv`  
**Description:** Contains registration profiles for 500 individuals signed up for an upcoming regional health education workshop. The dataset intentionally includes common real-world data anomalies to practice programmatic data cleaning, validation, and preparing geographic strings for geocoding.

### Table Schema Summary

| Column Name | Data Type (Raw) | Description | Expected Format / Valid Values | Known Data Quality Issues Included |
| :--- | :--- | :--- | :--- | :--- |
| **Participant_ID** | Integer | Unique identifier assigned sequentially to each workshop registration row. | `1001` to `1500` | None (used to track indexing and deduplication results). |
| **Name** | String / Object | Full name of the participant registration. | `[First Name] [Last Name]` | Contains structural duplicates (the same individual registered multiple times with identical data across columns). |
| **Age** | Float / Numeric | The self-reported age of the program participant. | Positive integers representing adult ages (`18` to `82`). | 1. Contains standard nulls (`NaN`).<br>2. Uses `-999` as a sentinel placeholder value for missing responses. |
| **Phone** | String / Object | Primary contact phone number for the registration. | 10-digit U.S. numbers. | 1. Inconsistent formats: mixed use of delimiters `(XXX) XXX-XXXX`, `XXX-XXX-XXXX`, and unformatted strings `XXXXXXXXXX`.<br>2. Missing entries (`NaN` and `-999`). |
| **Address** | String / Object | Street number and name provided for mailing or residence validation. | `[Street Number] [Street Name] [Suffix]` | Inconsistent suffix naming conventions (e.g., alternating between variations like `St`, `St.`, `Street`, `st`, `Rd`, or `Avenue`). |
| **City** | String / Object | The primary municipality name tied to the residence. | Standard Northeast/East Coast municipal names. | 1. Inconsistent typography or colloquialisms (e.g., `"Philly"` instead of `"Philadelphia"`).<br>2. Structural typos / spacing issues (e.g., `"NewYork"` instead of `"New York"`). |
| **State** | String / Object | Two-letter USPS postal abbreviation for the state code. | Standard US State codes (`NY`, `PA`, `NJ`, `MA`, etc.). | Uneven distributions (skewed intentionally with over 80% concentrated in `NY` and `PA`). |
| **Zip_Code** | Float / Numeric | The postal ZIP code corresponding to the physical address. | 5-digit string or numeric codes (e.g., starting with `100xx`, `191xx`). | 1. Truncated leading zeros due to automatic numeric parser detection (e.g., New England codes beginning with `0` reading incorrectly).<br>2. Missing values (`NaN`).<br>3. Sentinel placeholder values (`-999`). |
| **Workshop_Goal** | String / Object | Open-ended text response detailing what the user hopes to gain from the workshop. | Free-form character text strings. | 1. Inconsistent text casing and punctuation (e.g., `"get fit!"` vs `"To learn how to eat healthier"`).<br>2. Weak responses or raw flags representing a skipped question (e.g., `"N/A"`, `-999`, or `NaN`). |

## Session Setup

Run the following cell to import the necessary modules data into the notebook environment. The Pandas library will be our primary tool for working with out data, with some supporting help from other tools like numpy, matplotlib, and geopy.

If you are working in your own environment or are getting module errors in your notebook, make sure you have installed all the necessary requirements by running this command in your terminal:

```python
pip install -r requirements.txt
```

> Pro-tip: you can run bash commands from within your notebook environment by using the `!` operator in a code cell, like this:
>
> `!pip install -r requirements.txt`

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopy

survey_data = pd.read_csv("data/workshop_participants_east_coast.csv")

survey_data.head()

,Participant_ID,Name,Age,Phone,Address,City,State,Zip_Code,Workshop_Goal
0,1001,Jacob Taylor,53.0,(280) 635-2449,710 Broadway Rd,Portland,ME,4184.0,To understand nutrition labels
1,1002,Jacob Brown,58.0,(583) 256-8770,452 Oak St,Buffalo,NY,14257.0,get fit!
2,1003,Mason Thomas,26.0,(203) 493-8898,351 Hill Road,Albany,NY,12290.0,Want to lower my blood pressure.
3,1004,Michael Gonzalez,43.0,7582841440,458 Broadway Rd.,Buffalo,NY,14218.0,I want to start cooking more at home.
4,1005,Mason Miller,27.0,9305552438,736 Washington Road,Albany,NY,12247.0,To understand nutrition labels


## Using Exploratory Data Analysis to Find Data Quality Issues

**TODO: intro supporting title**

**TODO: Explain that we've imported the csv data into a Pandas dataframe**

The first thing we're going to do is generate some basic info about our dataframe using the `.info()` method. If we didn't have the data dictionary provided above, we could still use this method to give us some inital facts about the number of rows, columns, and datatypes of those columns:

In [5]:
survey_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Participant_ID  500 non-null    int64  
 1   Name            500 non-null    str    
 2   Age             485 non-null    float64
 3   Phone           445 non-null    str    
 4   Address         500 non-null    str    
 5   City            500 non-null    str    
 6   State           500 non-null    str    
 7   Zip_Code        453 non-null    float64
 8   Workshop_Goal   407 non-null    str    
dtypes: float64(2), int64(1), str(6)
memory usage: 35.3 KB


some methods for grabbing individual pieces of this report:

In [14]:
print("Number of rows and columns in the dataset:")
print(survey_data.shape) # get rows and columns
print("===========================================")

print("Column names in the dataset:")
print(survey_data.columns) # get column names
print("===========================================")

print("Data types of columns in the dataset:")
print(survey_data.dtypes) # get data types of columns
print("===========================================")

print("Number of missing values in each column:")
print(survey_data.isnull().sum()) # get number of missing values in each column
print("===========================================")


print("Summary statistics for numeric columns in the dataset:")
print(survey_data.describe()) # get summary statistics for numeric columns
print("===========================================")

Number of rows and columns in the dataset:
(500, 9)
Column names in the dataset:
Index(['Participant_ID', 'Name', 'Age', 'Phone', 'Address', 'City', 'State',
       'Zip_Code', 'Workshop_Goal'],
      dtype='str')
Data types of columns in the dataset:
Participant_ID      int64
Name                  str
Age               float64
Phone                 str
Address               str
City                  str
State                 str
Zip_Code          float64
Workshop_Goal         str
dtype: object
Number of missing values in each column:
Participant_ID     0
Name               0
Age               15
Phone             55
Address            0
City               0
State              0
Zip_Code          47
Workshop_Goal     93
dtype: int64
Summary statistics for numeric columns in the dataset:
       Participant_ID         Age      Zip_Code
count      500.000000  485.000000    453.000000
mean      1250.500000  -27.243299  13169.269316
std        144.481833  276.019071   5490.613177
min       

## Discussion
Take a look at the summary statistics and the datatype tables. What are some things you notice about the calculations? How might we fix them?

### A note about DataTypes in Pansas
**TODO: write a short blurb about the different types of datatypes in pandas**

we need to fix the datatypes for `participant_ID` and `ZIP_Cide` so that they can be treated as text values. We're also going to make 'Age' an int

In [28]:
fixed_dtypes = survey_data.fillna(0).astype({
    'Age': 'int64',
    'Participant_ID': 'str',
    'Zip_Code': 'str',
})

print(fixed_dtypes.dtypes) # check data types after conversion

display(fixed_dtypes.head())

print("Summary statistics for numeric columns after conversion:")
print(fixed_dtypes.describe()) # get summary statistics for numeric columns after conversion

Participant_ID       str
Name                 str
Age                int64
Phone             object
Address              str
City                 str
State                str
Zip_Code             str
Workshop_Goal     object
dtype: object


,Participant_ID,Name,Age,Phone,Address,City,State,Zip_Code,Workshop_Goal
0,1001,Jacob Taylor,53,(280) 635-2449,710 Broadway Rd,Portland,ME,4184.0,To understand nutrition labels
1,1002,Jacob Brown,58,(583) 256-8770,452 Oak St,Buffalo,NY,14257.0,get fit!
2,1003,Mason Thomas,26,(203) 493-8898,351 Hill Road,Albany,NY,12290.0,Want to lower my blood pressure.
3,1004,Michael Gonzalez,43,7582841440,458 Broadway Rd.,Buffalo,NY,14218.0,I want to start cooking more at home.
4,1005,Mason Miller,27,9305552438,736 Washington Road,Albany,NY,12247.0,To understand nutrition labels


Summary statistics for numeric columns after conversion:
              Age
count  500.000000
mean   -26.426000
std    271.878636
min   -999.000000
25%     29.000000
50%     47.000000
75%     65.250000
max     82.000000


Now we can see age is the only numeric column being calculated in the summary stats. The string conversion on `Zip_Code` still preserved the `.0` decimal place at the end, but we'll take care of that in the "String Functions" section.

## Data Issue #1: Missing Values

In [31]:
survey_data.isnull().sum().sort_values(ascending=False) # get number of missing values in each column sorted in descending order

Workshop_Goal     93
Phone             55
Zip_Code          47
Age               15
Participant_ID     0
Address            0
Name               0
State              0
City               0
dtype: int64

Strategies for dealing with missing data:
- filling in 0 or "NA" (easiest / skews data + leaves data messy)
- dropping rows (quickest / risk of significant data loss)
- backfilling / forward filling (more realistic data / problems if data isn't in logical order)
- filling in the mean (more realistic data / skews overall trends towards the middle)

Discussion - How would you choose to handle missing values for each of these columns?

- Participant ID: **Fill** - assign a random ID
- Name: **Fill:** - Assign a special term like "Anonymous" or "Declined to provide name"
- Age: Fill with the mean or 0
- Phone: fill with "N/A"
- Address: fill with "N/A"
- City: fill with "N/A" OR try to derive from address + state
- State: fill with "N/A" OR try to derive from address + city
- Zip Code: fill with "N/A" OR try to derive from address + city + state
- Workshop Goal: fill in "None"

In [ ]:
survey_data = survey_data.fillna({
    'Age': survey_data['Age'].median(),
    'Phone': 'None',
    'Zip_Code': 'Unknown',
    'Workshop_Goal': 'None',
})

## Data Issue #2: Duplicate Values

In [ ]:
survey_data.fillna({
    'Age': survey_data['Age'].median(),
    'Participant_ID': 'Unknown',
    'Zip_Code': 'Unknown'
}).astype({
    'Age': 'int64',
    'Participant_ID': 'str',
    'Zip_Code': 'str',
}).info() # check data types after conversion

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer.Replace or remove non-finite values or cast to an integer typethat supports these values (e.g. 'Int64'): Error while type casting for column 'Age'